<a href="https://colab.research.google.com/github/kawastony/Quadratic-Mechanism-Lens/blob/main/TIFA_figures_paper_withcolablink.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ============================================================
# Figure 3 for TIFA Paper 2
# Two panels:
#   Top:    alpha_eff(z) — effective lapse function
#   Bottom: H(z)*f_eff  — temperature hierarchy T_U/T_dS
# Save as: figure3_paper2.pdf
# Run: python figure3_paper2.py
# ============================================================

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams.update({
    'font.family':     'serif',
    'font.size':       12,
    'axes.labelsize':  13,
    'legend.fontsize': 10.5,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'text.usetex':     False,
    'figure.dpi':      150,
    'axes.linewidth':  1.2,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top':       True,
    'ytick.right':     True,
})

# ── TIFA parameters ──────────────────────────────────────────
feff     = 0.305
Omega_DE = 0.69
Omega_m  = 0.31
Omega_r  = 9.0e-5
H0       = 1.0

rho_DE_0 = 3.0 * H0**2 * Omega_DE
Lambda4  = 0.1 * rho_DE_0
phi_i    = 0.377 * np.pi * feff

def V(phi):
    return Lambda4 * (1.0 + np.cos(phi / feff))

def dV(phi):
    return -Lambda4 / feff * np.sin(phi / feff)

# ── ODE system ───────────────────────────────────────────────
def odes(lna, y):
    phi, phidot_H = y
    a = np.exp(lna)
    rho_m  = Omega_m * a**(-3)
    rho_r  = Omega_r * a**(-4)
    Vphi = V(phi)
    dVphi = dV(phi)
    denom  = 3.0 - 0.5 * phidot_H**2
    if denom <= 0:
        denom = 1e-10
    H2 = (rho_m + rho_r + Vphi) / denom
    H  = np.sqrt(max(H2, 1e-30))
    phidot  = phidot_H * H
    rho_phi = 0.5 * phidot**2 + Vphi
    rho_tot = rho_m + rho_r + rho_phi
    p_phi   = 0.5 * phidot**2 - Vphi
    p_tot   = rho_r / 3.0 + p_phi
    Hdot_H2 = -0.5 * (rho_tot + p_tot) / H2
    dphi_dlna = phidot / H
    ddotphi = -3.0 * H * phidot - dVphi
    d_phiH_dlna = (ddotphi / H - phidot_H * Hdot_H2 * H) / H
    return [dphi_dlna, d_phiH_dlna]

# ── Integrate ────────────────────────────────────────────────
a_i   = 1.0 / 5.0
lna_i = np.log(a_i)
lna_f = 0.0

H_i = np.sqrt(
    (Omega_m * a_i**(-3) + Omega_r * a_i**(-4) + V(phi_i)) / 3.0
)
phidot_i   = -dV(phi_i) / (3.0 * H_i)
phidot_H_i = phidot_i / H_i

sol = solve_ivp(
    odes, (lna_i, lna_f), [phi_i, phidot_H_i],
    t_eval=np.linspace(lna_i, lna_f, 600),
    method='DOP853', rtol=1e-9, atol=1e-11
)

a_arr  = np.exp(sol.t)
z_arr  = 1.0 / a_arr - 1.0
phi_a  = sol.y[0]
phiH_a = sol.y[1]

# ── Compute alpha_eff and H*f_eff ────────────────────────────
aeff_arr = np.zeros(len(z_arr))
Hfeff_arr = np.zeros(len(z_arr))
H_arr    = np.zeros(len(z_arr))

for i in range(len(z_arr)):
    phi = phi_a[i]
    xH  = phiH_a[i]
    a   = a_arr[i]
    Vphi = V(phi)
    rho_m = Omega_m * a**(-3)
    rho_r = Omega_r * a**(-4)
    denom = 3.0 - 0.5 * xH**2
    H2 = (rho_m + rho_r + Vphi) / max(denom, 1e-10)
    H  = np.sqrt(max(H2, 1e-30))
    H_arr[i] = H
    vphi = abs(xH)   # |phidot/H|
    # Note: vphi here is phidot/H, but we defined
    # v_phi = sqrt(2*rho_KE / 3H^2) = |phidot| / (H*sqrt(3))
    # Let us be precise:
    phidot = xH * H
    rho_KE = 0.5 * phidot**2
    vphi_correct = np.sqrt(2.0 * rho_KE / (3.0 * H2))
    aeff_arr[i]  = np.sqrt(max(1.0 - vphi_correct**2, 0))
    # H*feff in units of H0 (H0=1 here)
    Hfeff_arr[i] = H * feff

# Sort ascending z
idx     = np.argsort(z_arr)
z_plot  = z_arr[idx]
aeff_p  = aeff_arr[idx]
Hfeff_p = Hfeff_arr[idx]

# ── LCDM reference for alpha_eff ────────────────────────────
# In LCDM, phi=0, so vphi=0, alpha_eff=1 exactly
aeff_lcdm = np.ones(len(z_plot))

# ── Two-panel figure ─────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(7.5, 8.5), sharex=True
)
fig.subplots_adjust(hspace=0.08)

# ── Top panel: alpha_eff(z) ──────────────────────────────────
ax1.plot(
    z_plot, aeff_lcdm,
    color='gray', linewidth=1.4,
    linestyle=':', label=r'$\Lambda\mathrm{CDM} (\alpha_{\mathrm{eff}}=1)$'
)
ax1.plot(
    z_plot, aeff_p,
    color='#27AE60', linewidth=2.5,
    label=r'TIFA $\alpha_{\mathrm{eff}}(z) = \sqrt{1-v_\phi^2}$'
)

# Shade the deviation
ax1.fill_between(
    z_plot, aeff_p, aeff_lcdm,
    alpha=0.20, color='#27AE60',
    label=r'Deviation from $\alpha_{\mathrm{eff}}=1$'
)

ax1.set_ylabel(
    r'Effective lapse $\alpha_{\mathrm{eff}}(z)$',
    labelpad=8
)
ax1.set_ylim(0.955, 1.010)
ax1.set_xlim(0, 4)
ax1.legend(loc='lower right', framealpha=0.9)
ax1.grid(True, alpha=0.25, linestyle='--')
ax1.set_title(
    r'Observer $\mathcal{B}$ Geometry: '
    r'Lapse and Temperature Hierarchy',
    pad=10
)
ax1.annotate(
    r'$\alpha_{\mathrm{eff}}(z=0) \simeq 0.974$' + '\n'
    + r'$\approx 2.6\%$ below cosmic time',
    xy=(0.60, 0.12), xycoords='axes fraction',
    fontsize=10.5,
    bbox=dict(boxstyle='round,pad=0.3',
              facecolor='white', alpha=0.85)
)

# ── Bottom panel: H(z)*f_eff ─────────────────────────────────
# This is T_Unruh / T_dS = H*f_eff
# Scale: at z=0, H=H0=1 (code units), feff=0.305
# In physical units this is ~3.6e-62
# We plot the dimensionless ratio in code units
# and annotate with physical value

ax2.plot(
    z_plot, Hfeff_p,
    color='#8E44AD', linewidth=2.5,
    label=r'$H(z)\,f_{\mathrm{eff}} = T_{\mathrm{Unruh}}/T_{\mathrm{dS}}$'
)

# Mark z=0 value
ax2.axhline(
    Hfeff_p[0], color='#8E44AD',
    linewidth=1.0, linestyle='--', alpha=0.5
)

# Mark z=0 point
ax2.scatter(
    [z_plot[0]], [Hfeff_p[0]],
    color='#8E44AD', s=60, zorder=5
)

ax2.set_xlabel(r'Redshift $z$', labelpad=8)
ax2.set_ylabel(
    r'$H(z)\,f_{\mathrm{eff}}$ (code units, $H_0=1$)',
    labelpad=8
)
ax2.set_ylim(0, Hfeff_p[-1] * 1.25)
ax2.legend(loc='upper left', framealpha=0.9)
ax2.grid(True, alpha=0.25, linestyle='--')

ax2.annotate(
    r'$\frac{T_{\mathrm{Unruh}}}{T_{\mathrm{dS}}}\vert_{z=0}'
    r'= H_0 f_{\mathrm{eff}} \simeq 3.6\times10^{-62}$'
    + '\n' + r'(code units: $H_0 f_{\mathrm{eff}} = 0.305$)',
    xy=(0.04, 0.72), xycoords='axes fraction',
    fontsize=10.0,
    bbox=dict(boxstyle='round,pad=0.3',
              facecolor='white', alpha=0.85)
)

ax2.annotate(
    r'$H(z)\,f_{\mathrm{eff}} \ll 1$ in physical units',
    xy=(0.04, 0.15), xycoords='axes fraction',
    fontsize=10.5,
    color='#6C3483',
    bbox=dict(boxstyle='round,pad=0.3',
              facecolor='white', alpha=0.85)
)

# Panel labels
ax1.text(
    0.97, 0.06, r'(a)',
    transform=ax1.transAxes,
    fontsize=12, ha='right', va='bottom',
    fontweight='bold'
)
ax2.text(
    0.97, 0.06, r'(b)',
    transform=ax2.transAxes,
    fontsize=12, ha='right', va='bottom',
    fontweight='bold'
)

plt.savefig('figure3_paper2.pdf', bbox_inches='tight')
plt.savefig(
    'figure3_paper2.png', bbox_inches='tight', dpi=200
)
print("Saved: figure3_paper2.pdf and figure3_paper2.png")

Saved: figure3_paper2.pdf and figure3_paper2.png
